Snowflake Tables, Views, Sequences, Types & Indexing — Complete Reference

*Co-authored with CoCo*

---
## 1. Types of Tables in Snowflake

Snowflake supports **9 table types**, each designed for different workloads.

| # | Table Type | Persistence | Time Travel | Fail-safe | Use Case |
|---|-----------|-------------|-------------|-----------|----------|
| 1 | Permanent | Survives session | Up to 90 days | 7 days | Production data |
| 2 | Transient | Survives session | Up to 1 day | None | Staging/ETL |
| 3 | Temporary | Session only | Up to 1 day | None | Scratch/intermediate |
| 4 | External | Metadata only | None | None | Query files in cloud storage |
| 5 | Iceberg | Survives session | Yes (managed) | None | Open-format interoperability |
| 6 | Event | Survives session | Yes | Yes | Telemetry/logging |
| 7 | Dynamic | Auto-refreshed | Yes | Yes | Declarative pipelines |
| 8 | Hybrid | Survives session | Yes | Yes | Low-latency OLTP + OLAP |
| 9 | Interactive | Session-scoped | None | None | Parameterized app state |

---
### 1.1 Permanent Table (Default)

- Created by default with `CREATE TABLE`.
- Full Time Travel (up to 90 days on Enterprise+) and 7-day Fail-safe.
- Best for: **production, business-critical data**.

```sql
CREATE TABLE customers (
    customer_id INT AUTOINCREMENT,
    name VARCHAR(100) NOT NULL,
    email VARCHAR(255),
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);
```

---
### 1.2 Transient Table

- Persists across sessions but has **no Fail-safe** period.
- Time Travel limited to **0 or 1 day**.
- Reduces storage costs for non-critical data.
- Best for: **staging, ETL intermediate results**.

```sql
CREATE TRANSIENT TABLE staging_raw_events (
    event_id VARCHAR,
    payload VARIANT,
    loaded_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Verify it's transient
SHOW TABLES LIKE 'staging_raw_events';
-- Check the "is_transient" column in the output.
```

---
### 1.3 Temporary Table

- Exists **only for the duration of the session**.
- Invisible to other users/sessions.
- No Fail-safe; Time Travel up to 1 day (within the session).
- Best for: **session-scoped scratch work**.

```sql
CREATE TEMPORARY TABLE temp_cart (
    product_id INT,
    quantity INT,
    added_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Also valid:
CREATE TEMP TABLE temp_cart2 (id INT);

-- Automatically dropped when session ends.
```

---

### Clustering Key in Snowflake

---

#### What is a Clustering Key?

A clustering key is a column (or set of columns/expressions) that defines the **physical sort order** of data across micro-partitions. Rows with similar clustering key values are co-located in the same micro-partitions, enabling Snowflake to **skip irrelevant partitions** during queries (partition pruning).

---

#### Why It Matters: Micro-Partition Pruning

```text
Without clustering key (random distribution):

  Query: WHERE order_date = '2025-01-15'

  ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐
  │Scan │ │Scan │ │Scan │ │Scan │ │Scan │ │Scan │  ← scans ALL partitions
  └─────┘ └─────┘ └─────┘ └─────┘ └─────┘ └─────┘

With clustering key on order_date (sorted):

  Query: WHERE order_date = '2025-01-15'

  ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐ ┌─────┐
  │Skip │ │Skip │ │SCAN │ │Skip │ │Skip │ │Skip │  ← scans only 1 partition
  └─────┘ └─────┘ └─────┘ └─────┘ └─────┘ └─────┘
```

Snowflake stores min/max metadata per partition. When data is clustered, the min/max ranges are narrow and non-overlapping, so most partitions can be skipped.

---

#### Creating and Managing Clustering Keys

```sql
-- Define at table creation
CREATE TABLE sales (
    order_id NUMBER,
    order_date DATE,
    customer_id NUMBER,
    region VARCHAR
)
CLUSTER BY (order_date, region);

-- Add to existing table
ALTER TABLE sales CLUSTER BY (order_date, region);

-- Check clustering efficiency
SELECT SYSTEM$CLUSTERING_INFORMATION('sales', '(order_date, region)');

-- Remove clustering key
ALTER TABLE sales DROP CLUSTERING KEY;
```

---

#### Which Tables Support Clustering Keys?

| Table Type | Clustering Key Supported? | Notes |
|------------|:---:|-------|
| Permanent | Yes | Most common use case |
| Transient | Yes | |
| Temporary | Yes | |
| Materialized View | Yes | |
| External Table | No | Read-only metadata layer |
| Hybrid Table | **No** | Organized by PRIMARY KEY automatically |
| Dynamic Table | No | Managed by Snowflake's refresh engine |
| Iceberg Table | No | Uses Iceberg's own partition spec |

---

#### Is Clustering Enabled by Default?

**No.** If you create a table without `CLUSTER BY`:
- No clustering key is defined.
- Automatic Clustering does NOT apply.
- Data is stored in micro-partitions in **insertion order** (natural ordering).

Snowflake still maintains micro-partition metadata (min/max) regardless — but without a clustering key, those ranges may heavily overlap, reducing pruning effectiveness on large tables.

---

#### What is Automatic Clustering?

Automatic Clustering is a **background service** that continuously re-organizes micro-partitions to maintain the clustering key's sort order as new data arrives.

| Condition | Automatic Clustering Active? |
|-----------|:---:|
| Table has `CLUSTER BY` defined | Yes — runs automatically in background |
| Table has NO `CLUSTER BY` | No — nothing happens |
| Table is suspended (`ALTER TABLE ... SUSPEND RECLUSTER`) | No — paused |

**You do NOT need to:**
- Start it manually — it activates when you define a clustering key.
- Assign a warehouse — Snowflake uses serverless compute (billed separately).
- Schedule it — Snowflake decides when reclustering is beneficial.

```sql
-- Define clustering key → Automatic Clustering starts
ALTER TABLE sales CLUSTER BY (order_date);

-- Suspend if needed (stop background reclustering)
ALTER TABLE sales SUSPEND RECLUSTER;

-- Resume
ALTER TABLE sales RESUME RECLUSTER;
```

---

#### When to Use Clustering Keys

| Use clustering keys when... | Don't bother when... |
|-----------------------------|----------------------|
| Table is large (multi-TB) | Table is small (< 1 GB) |
| Queries frequently filter on specific columns | Queries scan most of the table anyway |
| Partition pruning is poor (check with `SYSTEM$CLUSTERING_INFORMATION`) | Table is write-heavy with rare reads |
| Existing natural ordering doesn't match query patterns | Data is already well-ordered by load sequence |

---

#### Best Practices for Choosing Clustering Columns

1. Choose columns that appear most in **WHERE** and **JOIN** conditions.
2. Prefer columns with **high cardinality but not unique** (dates, regions, categories).
3. Limit to **3–4 columns** maximum — more columns dilute effectiveness.
4. Put the most selective (most filtered) column first.

```sql
-- Good: date + region (common filters, moderate cardinality)
CLUSTER BY (order_date, region)

-- Bad: unique ID (every value is different — no pruning benefit)
CLUSTER BY (order_id)

-- Expression-based clustering is also supported
CLUSTER BY (DATE_TRUNC('month', order_date), region)
```

---

#### Cost of Clustering

Automatic Clustering consumes **serverless compute credits** (not your warehouse). It runs only when Snowflake determines reclustering is cost-effective.

```sql
-- Check reclustering credit usage
SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.AUTOMATIC_CLUSTERING_HISTORY
WHERE TABLE_NAME = 'SALES'
ORDER BY START_TIME DESC;
```

---

#### Summary

| Fact | Detail |
|------|--------|
| What it does | Physically sorts data across micro-partitions |
| Benefit | Fewer partitions scanned → faster queries |
| Defined by | `CLUSTER BY (col1, col2, ...)` |
| Enabled by default? | No — only when you define a clustering key |
| Maintenance | Automatic (serverless background service) |
| Cost | Serverless compute credits for reclustering |
| Supported tables | Permanent, Transient, Temporary, Materialized Views |
| Not supported | Hybrid, External, Dynamic, Iceberg |

---
### 1.4 External Table

- A **read-only metadata layer** over files in an external stage (S3, GCS, Azure Blob).
- Data remains in your cloud storage; Snowflake reads it on query.
- Supports partitioning and auto-refresh via cloud event notifications.
- Does not support DML operations.
- Only supports querying data
- Best for: **querying data lakes without loading into Snowflake**.

```sql
-- Step 1: Create a stage
CREATE STAGE my_ext_stage
  URL = 's3://my-bucket/data/'
  CREDENTIALS = (AWS_ROLE = 'arn:aws:iam::123456789:role/my-role');

-- Step 2: Create the external table
CREATE EXTERNAL TABLE ext_sales (
    sale_date DATE AS (VALUE:sale_date::DATE),
    product VARCHAR AS (VALUE:product::VARCHAR),
    amount NUMBER(10,2) AS (VALUE:amount::NUMBER(10,2))
)
WITH LOCATION = @my_ext_stage/sales/
FILE_FORMAT = (TYPE = PARQUET)
AUTO_REFRESH = TRUE;

-- Query it like a regular table
SELECT * FROM ext_sales WHERE sale_date > '2025-01-01';
```

---
### 1.5 Iceberg Table (Detailed)

Apache Iceberg is an **open table format** — data is stored as Parquet files with Iceberg metadata, making it readable by Spark, Flink, Trino, and other engines.

Snowflake supports Iceberg tables in **two catalog modes** and **two storage modes**.

---

#### Key Concepts: Volume (Mandatory) & Catalog

**Every Iceberg table requires a volume.** This is non-negotiable — Iceberg data must be stored somewhere, and the volume defines where. You either specify it explicitly in the `CREATE` statement or inherit it from account/schema/database defaults.

The volume comes in **two forms:**

| Volume Type | What It Is | Setup Required |
|-------------|------------|----------------|
| **Custom External Volume** | A user-created object pointing to your own cloud bucket (S3/GCS/Azure) | Yes — you create the EXTERNAL VOLUME object |
| **`SNOWFLAKE_MANAGED`** | A **system-defined (pre-built) external volume** that stores Iceberg files in **Snowflake's internal storage** | None — it exists out of the box |

---

#### How the Volume Parameter Works

| Scenario | What Happens |
|----------|-------------|
| `EXTERNAL_VOLUME = 'my_custom_vol'` | Data stored in your cloud bucket (defined in that volume) |
| `EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'` | Data stored in Snowflake's internal storage |
| `EXTERNAL_VOLUME` omitted in CREATE | Inherited from schema → database → account default (still mandatory — just implicit) |

**Setting defaults (so you can omit EXTERNAL_VOLUME in CREATE):**
```sql
-- Set default volume at account level
ALTER ACCOUNT SET DEFAULT_EXTERNAL_VOLUME = 'my_iceberg_vol';

-- Set default volume at database level
ALTER DATABASE my_db SET DEFAULT_EXTERNAL_VOLUME = 'my_iceberg_vol';

-- Set default volume at schema level
ALTER SCHEMA my_db.my_schema SET DEFAULT_EXTERNAL_VOLUME = 'my_iceberg_vol';
```

If no default is configured and you omit `EXTERNAL_VOLUME`, the CREATE statement **will fail**.

---

#### `BASE_LOCATION` — Optional (since 2025_01 bundle)

`BASE_LOCATION` defines the subfolder within the volume where data is written. It is **optional** for Snowflake-managed Iceberg tables:

| Scenario | What Happens |
|----------|-------------|
| `BASE_LOCATION` specified | Snowflake writes to `STORAGE_BASE_URL/<your_base_location>.<randomId>/` |
| `BASE_LOCATION` omitted | Snowflake auto-generates: `STORAGE_BASE_URL/<database>/<schema>/<table_name>.<randomId>/` |
| `BASE_LOCATION_PREFIX` set at schema | Snowflake uses: `STORAGE_BASE_URL/<prefix>/<table_name>.<randomId>/` |

---

#### Minimal CREATE Statements

```sql
-- Explicit volume + explicit location (most common)
CREATE ICEBERG TABLE events (
    event_id STRING,
    user_id INT
)
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'my_iceberg_vol'
  BASE_LOCATION = 'events/';

-- Managed storage, no bucket needed
CREATE ICEBERG TABLE events (
    event_id STRING,
    user_id INT
)
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED';

-- Defaults configured — simplest possible form
-- (account/schema default EXTERNAL_VOLUME must be set)
CREATE ICEBERG TABLE events (
    event_id STRING,
    user_id INT
)
  CATALOG = 'SNOWFLAKE';
```

---

#### Summary: What's Mandatory vs Optional

| Parameter | Required? | Notes |
|-----------|-----------|-------|
| `CATALOG` | Yes | Always required. `'SNOWFLAKE'` for managed, or catalog integration name for external |
| `EXTERNAL_VOLUME` | **Yes (always)** | Must be specified or inherited from defaults. Either your volume or `'SNOWFLAKE_MANAGED'` |
| `BASE_LOCATION` | No | Optional since 2025_01; auto-generated if omitted |

---
#### Mode A: Snowflake-Managed Catalog + Your Own Bucket

Snowflake manages the Iceberg catalog. Data lives in **your cloud storage**.

**Step 1:** Create an external volume pointing to your bucket.
```sql
CREATE OR REPLACE EXTERNAL VOLUME my_iceberg_vol
  STORAGE_LOCATIONS = (
    (
      NAME = 'my-s3-location'
      STORAGE_BASE_URL = 's3://my-bucket/iceberg-data/'
      STORAGE_PROVIDER = 'S3'
      STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::123456789:role/iceberg-role'
    )
  );
```

**Step 2:** Create the Iceberg table.
```sql
CREATE ICEBERG TABLE web_events (
    event_id STRING,
    user_id INT,
    event_type STRING,
    event_ts TIMESTAMP_NTZ
)
  CATALOG = 'SNOWFLAKE'               -- Snowflake manages the Iceberg catalog
  EXTERNAL_VOLUME = 'my_iceberg_vol'   -- Your bucket
  BASE_LOCATION = 'web_events/';       -- Optional subfolder
```

**Step 3:** Full DML works.
```sql
INSERT INTO web_events VALUES ('e1', 101, 'click', CURRENT_TIMESTAMP());
UPDATE web_events SET event_type = 'purchase' WHERE event_id = 'e1';
DELETE FROM web_events WHERE user_id = 101;
MERGE INTO web_events t USING new_events s ON t.event_id = s.event_id
  WHEN MATCHED THEN UPDATE SET event_type = s.event_type
  WHEN NOT MATCHED THEN INSERT VALUES (s.event_id, s.user_id, s.event_type, s.event_ts);
```

---
#### Mode B: Snowflake-Managed Catalog + Snowflake Storage (`SNOWFLAKE_MANAGED`)

The **simplest setup** — no bucket, no external volume creation needed. Snowflake stores everything internally in Iceberg format.

```sql
-- No external volume object creation needed!
CREATE ICEBERG TABLE simple_events (
    event_id STRING,
    user_id INT,
    payload VARIANT
)
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED';   -- System-defined volume → Snowflake storage
  -- BASE_LOCATION is optional (auto-generated)
```

**When to use `SNOWFLAKE_MANAGED`:**
- You want Iceberg format benefits (schema evolution, time travel, partition evolution).
- You do NOT need other engines (Spark/Flink) to directly read the raw files.
- You want zero infrastructure setup.

**Trade-off:** Data stored in Snowflake storage incurs **Snowflake storage costs** (unlike your-bucket mode where your cloud provider bills you directly).

---
#### Mode C: Externally-Managed Iceberg Table (External Catalog)

Another engine (Spark, Flink, Trino) writes the Iceberg data. Snowflake reads it via a **catalog integration**.

**Step 1:** Create a catalog integration.
```sql
-- Example: AWS Glue
CREATE CATALOG INTEGRATION glue_catalog
  CATALOG_SOURCE = GLUE
  CATALOG_NAMESPACE = 'analytics_db'
  TABLE_FORMAT = ICEBERG
  GLUE_AWS_ROLE_ARN = 'arn:aws:iam::123456789:role/glue-role'
  GLUE_CATALOG_ID = '123456789'
  ENABLED = TRUE;

-- Example: REST catalog (Polaris, Unity Catalog)
CREATE CATALOG INTEGRATION rest_catalog
  CATALOG_SOURCE = ICEBERG_REST
  CATALOG_URI = 'https://my-catalog.example.com/api'
  CATALOG_NAMESPACE = 'prod'
  TABLE_FORMAT = ICEBERG
  ENABLED = TRUE;
```

**Step 2:** Create the externally-managed Iceberg table.
```sql
CREATE ICEBERG TABLE ext_web_events
  EXTERNAL_VOLUME = 'my_iceberg_vol'
  CATALOG = 'glue_catalog'
  CATALOG_TABLE_NAME = 'web_events';
```

**Step 3:** Query (read-only for most external catalogs; write supported for REST catalogs).
```sql
SELECT * FROM ext_web_events WHERE event_ts > '2025-01-01' LIMIT 100;

-- Refresh metadata (pick up new snapshots written externally)
ALTER ICEBERG TABLE ext_web_events REFRESH;
```

---
#### Iceberg Table — Full Comparison

| Feature | Mode A: Your Bucket | Mode B: SNOWFLAKE_MANAGED | Mode C: External Catalog |
|---------|---------------------|--------------------------|-------------------------|
| **Storage location** | Your S3/GCS/Azure | Snowflake internal | Your S3/GCS/Azure |
| **Catalog manager** | Snowflake | Snowflake | External (Glue/REST/etc.) |
| **DML support** | Full | Full | Read-only (write for REST catalogs) |
| **Time Travel** | Yes | Yes | Limited (snapshot-based) |
| **Other engines can read?** | Yes (open Parquet) | No | Yes |
| **Setup complexity** | Medium (create ext vol) | Low (nothing extra) | High (catalog integration) |
| **`EXTERNAL_VOLUME` value** | Your custom volume name | `'SNOWFLAKE_MANAGED'` | Your custom volume name |
| **`CATALOG` value** | `'SNOWFLAKE'` | `'SNOWFLAKE'` | Catalog integration name |
| **`BASE_LOCATION` required?** | Optional (auto-gen) | Optional (auto-gen) | Not applicable |
| **Fail-safe** | None | None | None |
| **Storage cost** | Your cloud provider | Snowflake bills you | Your cloud provider |

#### Convert External → Managed
```sql
ALTER ICEBERG TABLE ext_web_events CONVERT TO MANAGED
  BASE_LOCATION = 'converted/web_events/';
```

---
### 1.6 Event Table

---

#### What is Telemetry Data?

Telemetry is **diagnostic data emitted by running code** — it tells you what your application did, how long it took, and whether anything went wrong. It is NOT business data; it exists for debugging, monitoring, and performance analysis.

| Type | What It Records | Example |
|------|----------------|---------|
| **Logs** | A message at a specific point in time | `"Processing batch 42"`, `"ERROR: null pointer"` |
| **Traces (Spans)** | The duration of an operation (start → end) | `"validate_input took 120ms"` |
| **Metrics** | A numeric measurement | `"queue_depth = 15"`, `"rows_processed = 10000"` |

---

#### What is an Event Table?

An event table is Snowflake's **storage destination for telemetry**. It has a fixed schema (you don't define columns), and Snowflake writes rows into it automatically when your instrumented code runs.

---

#### Step 1: Create an Event Table

```sql
CREATE EVENT TABLE my_db.my_schema.my_events;
```

No column definitions needed. Snowflake creates the table with these fixed columns:

| Column | Type | Stores |
|--------|------|--------|
| `TIMESTAMP` | TIMESTAMP_NTZ | When the event was emitted |
| `RECORD_TYPE` | VARCHAR | `'LOG'`, `'SPAN'`, or `'SPAN_EVENT'` |
| `RECORD` | VARIANT | Severity level, structured fields |
| `VALUE` | VARIANT | The actual log message or metric value |
| `RESOURCE_ATTRIBUTES` | VARIANT | Source: UDF name, warehouse, database, schema |
| `RECORD_ATTRIBUTES` | VARIANT | Extra key-value metadata |
| `SCOPE` | VARIANT | Logger name and version |

---

#### Step 2: Associate with a Scope

```sql
-- Account-wide: capture from all UDFs/procedures in the account
ALTER ACCOUNT SET EVENT_TABLE = 'my_db.my_schema.my_events';

-- Database-specific: overrides account-level for this database
ALTER DATABASE analytics_db SET EVENT_TABLE = 'my_db.my_schema.my_events';
```

Snowflake also provides a built-in default: `SNOWFLAKE.TELEMETRY.EVENTS` (active unless you override it).

---

#### Step 3: Set Log Level

```sql
ALTER DATABASE my_db SET LOG_LEVEL = 'INFO';       -- TRACE | DEBUG | INFO | WARN | ERROR | FATAL | OFF
ALTER DATABASE my_db SET TRACE_LEVEL = 'ON_EVENT'; -- OFF | ALWAYS | ON_EVENT
```

Only messages **at or above** the configured level are captured. Setting `'WARN'` means INFO and DEBUG are silently discarded.

---

#### Step 4: Instrument Your Code

You never INSERT rows manually. Your code calls a logging function → Snowflake intercepts it → a row appears in the event table.

**Python:**
```sql
CREATE OR REPLACE FUNCTION my_udf(x INT)
  RETURNS INT
  LANGUAGE PYTHON
  RUNTIME_VERSION = '3.9'
  HANDLER = 'compute'
AS $$
import logging
logger = logging.getLogger("my_udf")

def compute(x):
    logger.info(f"Processing: {x}")   # captured → event table
    print("hello")                     # NOT captured (stdout ignored)
    return x * 2
$$;
```

**SQL:**
```sql
CREATE OR REPLACE PROCEDURE process_orders()
  RETURNS VARCHAR
  LANGUAGE SQL
AS
BEGIN
  SYSTEM$LOG_INFO('Starting order processing');
  SYSTEM$LOG_WARN('Found 5 duplicate orders');
  RETURN 'Done';
END;
```

---

#### Step 5: Query the Event Table

```sql
SELECT
    TIMESTAMP,
    RECORD:severity_text::STRING AS severity,
    VALUE::STRING AS message,
    RESOURCE_ATTRIBUTES:"snow.executable.name"::STRING AS source_function
FROM my_db.my_schema.my_events
WHERE RECORD_TYPE = 'LOG'
  AND TIMESTAMP > DATEADD('hour', -1, CURRENT_TIMESTAMP())
ORDER BY TIMESTAMP DESC;
```

---

#### How Data Flows (End-to-End)

```text
  ┌──────────────────────────────────────────────────────────────────┐
  │  YOUR CODE (UDF / Procedure / Streamlit)                         │
  │                                                                  │
  │    logger.info("Processing order 42")                            │
  │         │                                                        │
  └─────────┼────────────────────────────────────────────────────────┘
            │
            ▼
  ┌──────────────────────────────────────────────────────────────────┐
  │  SNOWFLAKE RUNTIME (injected handler)                            │
  │                                                                  │
  │  Intercepts the logging call and serializes it:                  │
  │    TIMESTAMP        = 2025-07-30 10:15:03                        │
  │    RECORD_TYPE      = 'LOG'                                      │
  │    RECORD           = {"severity_text": "INFO"}                   │
  │    VALUE            = "Processing order 42"                      │
  │    RESOURCE_ATTRS   = {"snow.executable.name": "MY_UDF", ...}    │
  │         │                                                        │
  └─────────┼────────────────────────────────────────────────────────┘
            │
            ▼
  ┌──────────────────────────────────────────────────────────────────┐
  │  EVENT TABLE (my_db.my_schema.my_events)                         │
  │                                                                  │
  │  Row is stored. You query it with SELECT.                        │
  └──────────────────────────────────────────────────────────────────┘
```

---

#### Supported Libraries — How to Know What Works

Snowflake **injects a handler** into specific libraries at runtime. Only calls to these libraries are intercepted:

| Language | Supported Library | How Snowflake Hooks In |
|----------|-------------------|------------------------|
| Python | `logging` (standard library) | Attaches a custom handler to the root logger |
| Java/Scala | SLF4J API | Provides a Snowflake SLF4J backend at runtime |
| JavaScript | `snowflake.log()` | Built-in Snowflake JS API method |
| SQL | `SYSTEM$LOG_INFO/WARN/ERROR/FATAL` | Native Snowflake system functions |

**The rule:** If it's not in this table, it's not captured. Period.

---

#### When Does a Custom Logger Work?

A custom logging class is captured **only if** it delegates to a supported library internally.

```text
  ┌───────────────────────────────────────────────────────────────┐
  │  DECISION: Will my custom logger be captured?                  │
  │                                                                │
  │  Does your custom class call one of these internally?          │
  │    • Python: logging.getLogger(...).info/warn/error(...)       │
  │    • Java:   LoggerFactory.getLogger(...).info(...)            │
  │    • JS:     snowflake.log(...)                                │
  │    • SQL:    SYSTEM$LOG_*(...)                                 │
  │                                                                │
  │       YES → Captured (Snowflake's handler intercepts it)       │
  │       NO  → NOT captured (invisible to Snowflake)              │
  └───────────────────────────────────────────────────────────────┘
```

**Works (delegates to `logging`):**
```python
import logging

class MyAppLogger:
    def __init__(self, name):
        self._logger = logging.getLogger(name)  # hooks into standard library

    def info(self, msg):
        self._logger.info(f"[APP] {msg}")       # captured by Snowflake

    def error(self, msg):
        self._logger.error(f"[APP] {msg}")      # captured
```

**Does NOT work (bypasses `logging`):**
```python
class MyAppLogger:
    def info(self, msg):
        print(f"[LOG] {msg}")        # NOT captured — print is stdout

    def error(self, msg):
        with open("/tmp/err.log", "a") as f:
            f.write(msg)             # NOT captured — file I/O
```

---

#### How Data is Stored (Structure)

Each log/trace/metric becomes **one row** in the event table. Data is stored in **VARIANT columns** (semi-structured JSON). You access nested fields using Snowflake's `:` dot notation:

```text
┌────────────────────────────────────────────────────────────────────────────┐
│ One row in the event table:                                                │
├────────────────────┬───────────────────────────────────────────────────────┤
│ TIMESTAMP          │ 2025-07-30 10:15:03.456                              │
│ RECORD_TYPE        │ 'LOG'                                                │
│ RECORD             │ {"severity_text": "INFO", "severity_number": 9}       │
│ VALUE              │ "Processing order 42"                                 │
│ RESOURCE_ATTRIBUTES│ {"snow.executable.name": "MY_UDF",                    │
│                    │  "snow.database.name": "MY_DB",                       │
│                    │  "snow.schema.name": "PUBLIC",                         │
│                    │  "snow.warehouse.name": "COMPUTE_WH"}                 │
│ RECORD_ATTRIBUTES  │ {"custom_key": "custom_value"}                        │
│ SCOPE              │ {"name": "my_udf_logger", "version": ""}              │
└────────────────────┴───────────────────────────────────────────────────────┘
```

**Querying nested fields:**
```sql
-- Access severity from RECORD
RECORD:severity_text::STRING

-- Access source UDF name from RESOURCE_ATTRIBUTES
RESOURCE_ATTRIBUTES:"snow.executable.name"::STRING

-- Access the log message
VALUE::STRING
```

---

#### What Event Tables Do NOT Capture

| System Event | Where To Find It |
|--------------|-------------------|
| Queries executed | `ACCOUNT_USAGE.QUERY_HISTORY` |
| Logins | `ACCOUNT_USAGE.LOGIN_HISTORY` |
| Data access (who read what) | `ACCOUNT_USAGE.ACCESS_HISTORY` |
| Warehouse activity | `ACCOUNT_USAGE.WAREHOUSE_EVENTS_HISTORY` |
| Data loading (COPY INTO) | `ACCOUNT_USAGE.COPY_HISTORY` |

**No instrumented code = empty event table.** Associating an event table means "route telemetry from code that calls a supported logging function here" — NOT "monitor everything in this database."

---

### 1.7 Dynamic Table

A Dynamic Table is a table whose contents are **derived from a SQL query** and **automatically kept up-to-date** by Snowflake. You define *what* the result should look like; Snowflake handles *when* and *how* to refresh it.

- Defined by a **declarative SQL query** with a `TARGET_LAG`.
- Snowflake automatically refreshes the data to stay within the specified lag.
- No manual scheduling — Snowflake orchestrates the pipeline.
- Best for: **data pipelines, incremental transformations, replacing streams+tasks**.

---

#### Core Concept

```text
Traditional ETL:                    Dynamic Table:

  Source Table                        Source Table
       │                                   │
       ▼                                   ▼
  Stream + Task               Dynamic Table (query + TARGET_LAG)
       │                                   │
       ▼                                   ▼
  Stored Procedure                   Auto Refresh
       │                              (Snowflake manages)
       ▼
  Target Table
```

You replace procedural pipelines (streams, tasks, procedures) with a **single declarative definition**.

---

#### Step 1: Create a Dynamic Table

```sql
CREATE OR REPLACE DYNAMIC TABLE sales_summary
  TARGET_LAG = '5 minutes'
  WAREHOUSE = my_wh
AS
  SELECT
      product_id,
      SUM(amount) AS total_sales
  FROM sales
  GROUP BY product_id;
```

- `TARGET_LAG = '5 minutes'` — data will be at most 5 minutes behind the source.
- `WAREHOUSE = my_wh` — the warehouse used for refresh compute.
- The `AS` query defines the table's contents.

---

#### Step 2: Understand TARGET_LAG

`TARGET_LAG` defines how fresh the data should be. It is a **best-effort target**, not a strict SLA.

| Value | Meaning |
|-------|---------|
| `'5 minutes'` | Snowflake refreshes so data is no more than 5 minutes stale |
| `'1 hour'` | Data may be up to 1 hour behind the source |
| `DOWNSTREAM` | Refreshes only when a downstream dynamic table requests updated data |

```sql
-- Time-based lag
CREATE DYNAMIC TABLE dt_orders
  TARGET_LAG = '10 minutes'
  WAREHOUSE = etl_wh
AS SELECT * FROM raw_orders;

-- Downstream-driven lag (lazy refresh)
CREATE DYNAMIC TABLE dt_intermediate
  TARGET_LAG = DOWNSTREAM
  WAREHOUSE = etl_wh
AS SELECT * FROM dt_orders WHERE status = 'ACTIVE';
```

---

#### Step 3: Build Chained Pipelines (Dependency Management)

Dynamic tables can depend on other dynamic tables. Snowflake **automatically manages refresh order**.

```sql
-- Layer 1: Cleansed data
CREATE DYNAMIC TABLE dt_orders
  TARGET_LAG = '10 minutes'
  WAREHOUSE = etl_wh
AS
  SELECT * FROM raw_orders WHERE amount > 0;

-- Layer 2: Aggregation (depends on Layer 1)
CREATE DYNAMIC TABLE dt_order_summary
  TARGET_LAG = '15 minutes'
  WAREHOUSE = etl_wh
AS
  SELECT customer_id, COUNT(*) AS order_count
  FROM dt_orders
  GROUP BY customer_id;

-- Layer 3: Business logic (depends on Layer 2)
CREATE DYNAMIC TABLE dt_vip_customers
  TARGET_LAG = '30 minutes'
  WAREHOUSE = etl_wh
AS
  SELECT customer_id, order_count
  FROM dt_order_summary
  WHERE order_count > 100;
```

```text
Pipeline flow:

  raw_orders → dt_orders → dt_order_summary → dt_vip_customers → Dashboard
                  10 min        15 min              30 min
```

Snowflake ensures Layer 1 refreshes before Layer 2, and Layer 2 before Layer 3.

---

#### Step 4: Understand Refresh Modes

Snowflake automatically chooses the refresh strategy:

| Mode | How It Works | When Used |
|------|--------------|-----------|
| **Incremental** | Processes only changed rows since last refresh | Preferred; used when the query supports it |
| **Full** | Recomputes the entire result from scratch | Fallback when incremental isn't possible |

You don't choose the mode — Snowflake decides based on the query pattern. Check which mode is in use:

```sql
SHOW DYNAMIC TABLES;
-- Look at the "refresh_mode" column

-- Or check refresh history
SELECT * FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(
    NAME => 'sales_summary'
));
```

---

#### Step 5: Common Use Cases

| Use Case | Example |
|----------|---------|
| Data warehouse layering | Raw → Cleansed → Aggregated → Dashboard |
| Aggregations / rollups | Monthly sales, daily active users |
| CDC-based transformations | Process data from Snowpipe or Streams |
| Replace Stream + Task pipelines | Simpler, declarative alternative |

```sql
-- Monthly sales aggregation
CREATE DYNAMIC TABLE monthly_sales
  TARGET_LAG = '1 hour'
  WAREHOUSE = etl_wh
AS
  SELECT
      DATE_TRUNC('month', order_date) AS month,
      SUM(amount) AS total_sales
  FROM orders
  GROUP BY 1;
```

---

#### Dynamic Table vs Materialized View vs Stream + Task

| Feature | Dynamic Table | Materialized View | Stream + Task |
|---------|--------------|-------------------|---------------|
| User-defined refresh lag | Yes (`TARGET_LAG`) | No (auto) | Yes (CRON schedule) |
| Complex SQL (JOINs, subqueries) | Yes | No (single table only) | Yes |
| Build multi-step pipelines | Yes (chaining) | No | Yes (manual wiring) |
| Incremental processing | Yes (automatic) | Yes (automatic) | Yes (manual via streams) |
| Setup complexity | Low (declarative) | Low | High (procedural) |
| Dependency management | Automatic | N/A | Manual |
| Best for | Transformation pipelines | Simple pre-computed aggregations | Complex workflow logic |

---

#### Benefits and Limitations

**Benefits:**
- Less ETL code — no tasks, no procedures, no scheduling
- Automatic incremental updates when possible
- Managed dependency tracking across chained tables
- Easier to maintain than Stream + Task pipelines

**Limitations:**
- Consumes warehouse credits during each refresh
- Not every SQL pattern supports incremental refresh (falls back to full)
- Refresh timing is best-effort, not a guaranteed SLA
- Cannot replace all orchestration (e.g., conditional branching, external API calls)

---

#### One-Line Summary

A Dynamic Table is an **automatically maintained, declarative, incrementally-refreshed table** defined by a query — enabling data pipelines without manual scheduling or procedural code.

---
### 1.8 Hybrid Table

A Hybrid Table is a Snowflake table designed for **both OLTP and OLAP** workloads in a single table. It maintains two storage layers (row-store + columnar) kept in sync automatically.

---

#### Background: OLTP vs OLAP

| | OLTP | OLAP |
|--|------|------|
| **Full form** | Online Transaction Processing | Online Analytical Processing |
| **Operations** | INSERT, UPDATE, DELETE single rows | SELECT with aggregations, scans, JOINs |
| **Access pattern** | Point lookup by key (one row) | Full/range scans over millions of rows |
| **Latency** | Milliseconds | Seconds to minutes |
| **Example** | "Update order #9999 to SHIPPED" | "Total revenue by region last quarter" |
| **Best Snowflake table** | Hybrid Table | Regular Permanent Table |

---

#### How a Hybrid Table Works

It maintains **two storage formats simultaneously**:

```text
┌──────────────────────────────────────────────────────┐
│                  HYBRID TABLE                         │
├──────────────────────────────────────────────────────┤
│                                                      │
│  Row Store (OLTP layer)       Columnar Store (OLAP)  │
│  ┌────────────────┐          ┌───────────────────┐   │
│  │ Primary Key Idx│          │ Micro-partitions  │   │
│  │ Secondary Idx  │   sync   │ (compressed cols) │   │
│  │                │ ◄──────► │                   │   │
│  │ Fast:          │          │ Fast:             │   │
│  │ - Point lookup │          │ - Full scans      │   │
│  │ - Single-row   │          │ - GROUP BY        │   │
│  │   INSERT/UPDATE│          │ - JOINs           │   │
│  │ - DELETE by PK │          │ - Aggregations    │   │
│  └────────────────┘          └───────────────────┘   │
└──────────────────────────────────────────────────────┘
```

- **OLTP queries** (WHERE pk = value) → routed to the row-store index → millisecond response.
- **OLAP queries** (GROUP BY, SUM, scans) → routed to columnar storage → standard Snowflake performance.
- Snowflake keeps both layers in sync transparently.

---

#### Creating a Hybrid Table

A `PRIMARY KEY` is **mandatory** — it drives the row-store index.

```sql
CREATE HYBRID TABLE orders (
    order_id VARCHAR PRIMARY KEY,              -- mandatory; creates primary index
    customer_id INT NOT NULL,
    status VARCHAR DEFAULT 'PENDING',
    amount DECIMAL(10,2),
    created_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    INDEX idx_customer (customer_id),          -- secondary index (optional)
    INDEX idx_status (status)                  -- secondary index (optional)
);
```

**Indexes in Hybrid Tables:**

| Index Type | How Created | Purpose |
|-----------|-------------|--------|
| Primary index | Automatically from `PRIMARY KEY` | Fast lookup by PK |
| Unique index | Automatically from `UNIQUE` constraint | Enforce uniqueness + fast lookup |
| Secondary index | Explicitly with `INDEX name (col)` | Fast lookup on non-PK columns |

---

#### Using a Hybrid Table: OLTP Operations

These use the **row-store index** — millisecond latency:

```sql
-- Point lookup by primary key
SELECT * FROM orders WHERE order_id = 'ORD-9999';

-- Single-row update
UPDATE orders SET status = 'SHIPPED' WHERE order_id = 'ORD-9999';

-- Single-row insert
INSERT INTO orders VALUES ('ORD-10000', 42, 'PENDING', 299.99, CURRENT_TIMESTAMP());

-- Delete by primary key
DELETE FROM orders WHERE order_id = 'ORD-10000';

-- Lookup via secondary index
SELECT * FROM orders WHERE customer_id = 42;
```

---

#### Using a Hybrid Table: OLAP Operations

These use the **columnar storage** — scans large data:

```sql
-- Aggregation
SELECT status, COUNT(*), SUM(amount) FROM orders GROUP BY status;

-- Time-based analysis
SELECT DATE_TRUNC('month', created_at) AS month, SUM(amount)
FROM orders GROUP BY 1 ORDER BY 1;

-- Analytical filtering
SELECT customer_id, AVG(amount)
FROM orders
GROUP BY customer_id
HAVING COUNT(*) > 10;
```

---

#### Constraint Enforcement: Hybrid Table vs Regular Table

In regular Snowflake tables, constraints are **NOT enforced** (metadata hints only). In Hybrid Tables, they are **actually enforced at write time**.

| Behavior | Regular Table | Hybrid Table |
|----------|---------------|---------------|
| Duplicate PK on INSERT | Allowed (no error) | **Rejected** |
| NULL in PK column | Allowed | **Rejected** |
| UNIQUE constraint | Not enforced | **Enforced** |
| FOREIGN KEY | Not enforced | **Enforced** |
| NOT NULL | Enforced | Enforced |

**Why the difference:**
- **Regular tables** prioritize bulk load throughput (COPY INTO, Snowpipe). Checking uniqueness row-by-row during parallel loads would kill performance. Constraints exist as optimizer hints.
- **Hybrid Tables** serve OLTP workloads where applications expect database-level integrity (like PostgreSQL or MySQL).

**Demonstration:**
```sql
-- Regular table: duplicates silently allowed
CREATE TABLE t_regular (id INT PRIMARY KEY, val INT);
INSERT INTO t_regular VALUES (1, 100);
INSERT INTO t_regular VALUES (1, 200);  -- succeeds (both rows exist)

-- Hybrid table: duplicates rejected
CREATE HYBRID TABLE t_hybrid (id INT PRIMARY KEY, val INT);
INSERT INTO t_hybrid VALUES (1, 100);   -- succeeds
INSERT INTO t_hybrid VALUES (1, 200);   -- ERROR: duplicate key violation
```

---

#### Full Comparison: Regular Table vs Hybrid Table

| Feature | Regular (Permanent) Table | Hybrid Table |
|---------|--------------------------|---------------|
| Storage format | Columnar only | Row-store + Columnar |
| Optimized for | OLAP (scans, aggregations) | OLTP + OLAP |
| PRIMARY KEY | Optional, not enforced | **Mandatory, enforced** |
| UNIQUE / FK | Not enforced | **Enforced** |
| Secondary indexes | Not supported | Supported |
| Clustering keys | Supported | Not supported |
| Search Optimization | Supported | Not needed (has indexes) |
| Time Travel | Up to 90 days | Up to 1 day |
| Fail-safe | 7 days | 7 days |
| Bulk load speed | Excellent | Slower (maintains both stores) |
| Point lookup speed | Slow (partition scan) | Fast (index, milliseconds) |
| Analytical scan speed | Excellent | Good (columnar store) |

---

#### Decision Guide

| Scenario | Recommendation |
|----------|----------------|
| Real-time point lookups + analytics on same data | Hybrid Table |
| High-frequency single-row writes + occasional reporting | Hybrid Table |
| Application needs enforced PK/UNIQUE/FK | Hybrid Table |
| Primarily analytics (multi-TB scans), rare single-row access | Regular Table + Search Optimization |
| High-volume bulk loads (COPY INTO) + heavy analytics | Regular Table |
| Need clustering keys for partition pruning | Regular Table |
| Need Time Travel > 1 day | Regular Table |
| 90%+ OLAP, 10% point lookups | Regular Table + Search Optimization |
| 90%+ OLTP, 10% analytics | Hybrid Table |

---
### 1.9 Interactive Table & Interactive Warehouse

---

#### What is an Interactive Table?

Per Snowflake official documentation, an interactive table is a **persistent** Snowflake table optimized for **low-latency, high-concurrency** workloads. It is NOT session-scoped — it survives sessions like a permanent table.

> **From docs:** "A type of Snowflake table that's optimized for low latency, high concurrency workloads that works well with interactive warehouses and can be used with standard Snowflake warehouses."

| Property | Value |
|----------|-------|
| Persistence | **Permanent** (survives sessions, like a regular table) |
| Optimized for | Low-latency reads, high concurrency |
| DML | Full (INSERT, UPDATE, DELETE) |
| Works with regular warehouse? | Yes — but with standard latency |
| Works with interactive warehouse? | Yes — **best performance** |
| Availability | Generally available (select AWS, GCP, Azure regions) |

---

#### What is an Interactive Warehouse?

A **specialized warehouse** with a query engine optimized for low-latency, high-concurrency queries. Designed to pair with interactive tables.

| Feature | Standard Warehouse | Interactive Warehouse |
|---------|-------------------|----------------------|
| Query engine | General-purpose (batch + analytics) | Optimized for low-latency, high concurrency |
| Best for | Large scans, complex JOINs, ETL | Dashboard queries, API serving, agentic AI |
| Works with interactive tables? | Yes (standard speed) | **Yes (fastest — ms latency)** |
| Works with regular tables? | Yes | Yes |
| Concurrency | Standard limits | Thousands of simultaneous queries |

---

#### Why Use an Interactive Warehouse with Interactive Tables?

Interactive tables work on regular warehouses, but **the interactive warehouse has a different internal query engine** specifically optimized for:
- Repetitive query shapes (same query, different parameters)
- Thousands of concurrent users
- Predictable, consistent millisecond latency
- Low cost per query

```text
  Interactive Table + Interactive Warehouse  → ms latency, thousands of QPS
  Interactive Table + Standard Warehouse     → works, but seconds-level latency
  Regular Table     + Interactive Warehouse  → works, but table not optimized
  Regular Table     + Standard Warehouse     → standard Snowflake behavior
```

The **best performance** comes from using both together.

---

#### Creating an Interactive Warehouse

```sql
CREATE INTERACTIVE WAREHOUSE my_interactive_wh
  WAREHOUSE_SIZE = 'SMALL';

-- Manage like any warehouse
ALTER INTERACTIVE WAREHOUSE my_interactive_wh RESUME;
ALTER INTERACTIVE WAREHOUSE my_interactive_wh SUSPEND;
```

---

#### Creating an Interactive Table

```sql
CREATE INTERACTIVE TABLE dashboard_metrics (
    metric_id INT,
    metric_name VARCHAR,
    region VARCHAR,
    current_value DECIMAL(10,2),
    updated_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Load data
INSERT INTO dashboard_metrics VALUES
  (1, 'active_users', 'US', 15230, CURRENT_TIMESTAMP()),
  (2, 'active_users', 'EU', 8420, CURRENT_TIMESTAMP()),
  (3, 'revenue_today', 'US', 142500.00, CURRENT_TIMESTAMP());

-- Query (use interactive warehouse for best performance)
USE WAREHOUSE my_interactive_wh;

SELECT metric_name, SUM(current_value)
FROM dashboard_metrics
WHERE region = 'US'
GROUP BY metric_name;
```

---

#### Summary

| Concept | One-Liner |
|---------|----------|
| Interactive Table | A **persistent** table optimized for fast, concurrent reads |
| Interactive Warehouse | A specialized warehouse with a low-latency query engine |
| Together | Millisecond latency at thousands of QPS |
| Apart | Still works, but without the full performance benefit |

---
## 2. Types of Views in Snowflake

| # | View Type | Stores Data? | Definition Visible? | Scope | Use Case |
|---|-----------|:---:|:---:|-------|----------|
| 1 | Standard View | No | Yes | Permanent | Simplify queries |
| 2 | Materialized View | Yes (cached) | Yes | Permanent | Expensive aggregations |
| 3 | Secure View | No | Hidden | Permanent | Data sharing, security |
| 4 | Temporary View | No | Yes | Session | Session scratch queries |
| 5 | Semantic View | No | Metadata-driven | Permanent | AI/Analyst natural-language queries |

> **Note:** Secure View is not a standalone view type — it is an **enhancement** that can be applied on top of Standard Views or Materialized Views.

---

### 2.1 Standard View

A saved SQL query that is **re-executed on every access**. Always returns current data.

- Definition visible to users with appropriate privileges.
- Persists in Database Explorer.
- No storage cost (does not store results).

```sql
CREATE VIEW active_customers AS
  SELECT customer_id, name, email
  FROM customers
  WHERE status = 'ACTIVE';

-- Query it
SELECT * FROM active_customers;

-- View the definition
SELECT GET_DDL('VIEW', 'active_customers');
```

---

### 2.2 Materialized View

Snowflake **pre-computes and stores** the query result. Automatically refreshed when base data changes. Faster reads but incurs extra storage and background maintenance compute.

```sql
CREATE MATERIALIZED VIEW mv_daily_sales AS
  SELECT
    DATE_TRUNC('day', sale_date) AS day,
    product_id,
    SUM(quantity) AS total_qty,
    SUM(revenue) AS total_revenue
  FROM sales
  GROUP BY 1, 2;

SELECT * FROM mv_daily_sales WHERE day >= '2025-01-01';
```

**Limitations:**
- Single base table only (no JOINs)
- No subqueries, UDFs, window functions, or HAVING
- Base table cannot be temporary or transient
- Incurs background maintenance costs

---

### 2.3 Secure View

An **enhancement** applied on top of a Standard View or Materialized View. The query definition is **hidden** from consumers, and the optimizer bypasses certain optimizations to prevent data leakage through query plans.

- Essential for **data sharing** and row-level security patterns.
- Consumers can query the view but cannot see its SQL definition (even with `GET_DDL`).

```sql
-- Secure standard view
CREATE SECURE VIEW customer_summary AS
  SELECT region, COUNT(*) AS customer_count
  FROM customers
  GROUP BY region;

-- Secure materialized view
CREATE SECURE MATERIALIZED VIEW secure_mv_sales AS
  SELECT product_id, SUM(revenue) AS total
  FROM sales
  GROUP BY product_id;
```

---

### 2.4 Temporary View

Exists only for the **current session**. Not visible to other users or sessions. Visible in Database Explorer only during the active session.

```sql
CREATE TEMPORARY VIEW temp_high_value_orders AS
  SELECT * FROM orders WHERE amount > 1000;

SELECT customer_id, COUNT(*) FROM temp_high_value_orders GROUP BY 1;
-- Automatically dropped when session ends.
```

---

### 2.5 Semantic View

A **metadata-driven layer** that maps business concepts (dimensions, measures) to SQL. Used by **Cortex Analyst** to translate natural-language questions into executable SQL.

Defined via YAML-based specifications — not a traditional SQL view.

```sql
CREATE SEMANTIC VIEW revenue_analytics
  FROM sales_table
  COLUMNS (
    order_date DIMENSION,
    region DIMENSION,
    revenue MEASURE AGGREGATION SUM,
    order_count MEASURE AGGREGATION COUNT
  )
  COMMENT = 'Revenue analytics for Cortex Analyst';
```

Reference: https://docs.snowflake.com/en/user-guide/views-semantic/sql

---
## 3. Sequences in Snowflake

A sequence is a **schema-level object** that generates unique numeric values. It is independent of any table — you create it once and reference it wherever needed.

---

### Step 1: Where is a Sequence Stored?

A sequence has **two parts** that live in different places:

| Part | Where It Lives | Persistent? | What It Contains |
|------|---------------|:-----------:|------------------|
| **Sequence definition + counter** | Cloud Services layer | Yes (always on, survives everything) | Name, schema, START, INCREMENT, ORDER/NOORDER, and the **current counter position** |
| **Cached batch (For NOORDER sequence only)** | Warehouse RAM | No (lost on suspend/crash) | A pre-fetched chunk of values for fast local access |

```text
┌──────────────────────────────────────────────────────────────────────┐
│  CLOUD SERVICES LAYER (always running)                               │
│                                                                      │
│  Stores permanently:                                                 │
│    • Sequence definition (name, schema, START, INCREMENT, ORDER)     │
│    • Counter position = the next value to hand out                   │
│                                                                      │
│  This is the SINGLE SOURCE OF TRUTH for the sequence.                │
│  It never goes down. Warehouse on or off — doesn't matter.           │
└───────────────────────────────────┬──────────────────────────────────┘
                                    │
                  NEXTVAL called    │
                                    ▼
┌──────────────────────────────────────────────────────────────────────┐
│  WAREHOUSE (compute layer — only active when running queries)        │
│                                                                      │
│  NOORDER mode:                                                       │
│    • Warehouse requests a batch from Cloud Services                  │
│    • Cloud Services advances counter and hands over the batch        │
│    • Warehouse holds batch in RAM (temporary, volatile)              │
│    • Each NEXTVAL reads from RAM — fast, no network call             │
│    • If warehouse suspends → RAM cleared → unused batch values LOST  │
│                                                                      │
│  ORDER mode:                                                         │
│    • Warehouse does NOT hold any cached values                       │
│    • Each NEXTVAL makes a network call to Cloud Services             │
│    • Cloud Services increments counter and returns one value         │
│    • Nothing stored in warehouse — nothing to lose                   │
└──────────────────────────────────────────────────────────────────────┘
```

**To be precise:**
- The **sequence itself** (definition + counter) is stored permanently in Cloud Services.
- The warehouse does NOT store the sequence. It temporarily **caches a batch of values in RAM** (NOORDER only). This cache is volatile — it vanishes on suspend/crash and is never persisted to disk.

---

### Step 2: The Two Generation Modes

| | NOORDER (default) | ORDER |
|--|-------------------|-------|
| **How it works** | Cloud Services gives warehouse a batch upfront. Warehouse serves values from local RAM. | Each NEXTVAL is a live request to Cloud Services. No batch, no cache. |
| **What warehouse holds** | A temporary batch in RAM (volatile) | Nothing |
| **Performance** | Fast (local RAM read, no network per call) | Slower (network round-trip per call) |
| **Ordering** | Unique only (not chronologically ordered across sessions) [monotonicity is not guaranteed] | Unique + strictly monotonic |
| **Gap risk** | High (unused batch lost on suspend + rollbacks) | Low (rollbacks only) |
| **Configuration** | Default (no keyword needed) | Must specify `ORDER` |

---

### Step 3: Does Suspending a Warehouse Lose Cached Sequence Values?

**Yes (NOORDER) / No (ORDER).** Here's the complete picture:

| Event | NOORDER (batch cached in RAM) | ORDER (no cache) |
|-------|-------------------------------|-------------------|
| Warehouse suspends (AUTO_SUSPEND or manual) | **Cached values LOST.** RAM is deallocated. | **Nothing lost.** No cache exists. |
| Warehouse crashes | **Cached values LOST.** Same effect as suspend. | **Nothing lost.** |
| Warehouse resizes (scale up/down) | **Cached values LOST.** Nodes are replaced. | **Nothing lost.** |
| Warehouse resumes | Requests a **new batch** from Cloud Services. | Continues one-at-a-time calls. |
| Counter in Cloud Services after suspend | **NOT rolled back.** Those values are gone permanently. | Accurate (reflects exact last value assigned). |

**Why Cloud Services does NOT roll back the counter:**

When Cloud Services hands batch [1–100] to the warehouse, it immediately advances its counter to 101. From Cloud Services' perspective, those 100 values are "given away" — it has no way to know which ones the warehouse actually used before it suspended. Rolling back would risk reissuing values that were already committed to tables by earlier queries.

**Practical implication:** If your warehouse has `AUTO_SUSPEND = 60` (suspends after 60 seconds of inactivity), and you use NOORDER sequences, you will see gaps every time the warehouse auto-suspends between bursts of inserts.

**If gaps are unacceptable:** Use `ORDER` mode. It has no cache, so suspend/resume never causes gaps (only rollbacks do).

---

### Step 4: Creating Sequences

```sql
-- Fast (NOORDER, default) — batch cached in warehouse RAM
CREATE SEQUENCE order_seq
  START = 1
  INCREMENT = 1
  COMMENT = 'Generates unique order IDs';

-- Strict ordering (ORDER) — no caching, one-at-a-time
CREATE SEQUENCE audit_seq
  START = 1
  INCREMENT = 1
  ORDER;

-- Large-step
CREATE SEQUENCE batch_seq START = 0 INCREMENT = 100;
```

---

### Step 5: Sequence Options

| Option | Description | Default |
|--------|-------------|--------|
| `START` | First value produced | 1 |
| `INCREMENT` | Step between values | 1 |
| `NOORDER` | Batch pre-fetched to warehouse RAM (fast, unique only) | Default |
| `ORDER` | One-at-a-time from Cloud Services (slower, strictly ordered) | Must specify |

---

### Step 6: Using Sequences

```sql
-- Get next value
SELECT order_seq.NEXTVAL;

-- Use in INSERT
INSERT INTO orders (order_id, customer_id, amount)
  VALUES (order_seq.NEXTVAL, 42, 199.99);

-- Use as column default
CREATE TABLE invoices (
    invoice_id INT DEFAULT order_seq.NEXTVAL,
    customer_id INT,
    amount DECIMAL(10,2)
);

-- Bulk insert (each row gets a unique value)
INSERT INTO invoices (customer_id, amount)
  SELECT customer_id, total_due FROM pending_invoices;
```

> **Note:** Snowflake does NOT support `CURRVAL`. Only `NEXTVAL` exists.

```sql
CREATE OR REPLACE SEQUENCE seq_5 START = 1 INCREMENT = 5;
SELECT seq_5.nextval a, seq_5.nextval b, seq_5.nextval c, seq_5.nextval d; -- 1, 6, 11, 16
SELECT seq_5.nextval a, seq_5.nextval b, seq_5.nextval c, seq_5.nextval d; -- 501, 506, 511, 516
SELECT seq_5.nextval a, seq_5.nextval b, seq_5.nextval c, seq_5.nextval d; -- 36, 41, 46, 51
SELECT seq_5.nextval a, seq_5.nextval b, seq_5.nextval c, seq_5.nextval d; -- 71, 76, 81, 86
```
> **Note:** Since, It is noorder sequence. Uniqueness is guaranteed, but monotonocity is not guaranteed.

---

### Step 7: Sequence vs AUTOINCREMENT

| Feature | Sequence | AUTOINCREMENT |
|---------|----------|---------------|
| Scope | Schema-level (reusable across tables) | Column-level (tied to one table) |
| Shared across tables? | Yes | No |
| Configurable after creation? | Yes (ALTER SEQUENCE) | No |
| Explicit reference needed? | Yes (`seq.NEXTVAL`) | No (automatic) |

```sql
-- AUTOINCREMENT: simple, one table
CREATE TABLE customers (
    id INT AUTOINCREMENT START 1 INCREMENT 1,
    name VARCHAR
);

-- Sequence: shared across tables (globally unique)
CREATE SEQUENCE shared_id START = 1 INCREMENT = 1;
CREATE TABLE orders (id INT DEFAULT shared_id.NEXTVAL, amount DECIMAL);
CREATE TABLE returns (id INT DEFAULT shared_id.NEXTVAL, reason VARCHAR);
```

---

### Step 8: Managing Sequences

```sql
ALTER SEQUENCE order_seq SET INCREMENT = 5;
ALTER SEQUENCE order_seq SET ORDER;
ALTER SEQUENCE order_seq SET NOORDER;
DROP SEQUENCE order_seq;
SHOW SEQUENCES;
```

---

### Step 9: Important Facts

| Fact | Detail |
|------|--------|
| Stored where? | **Cloud Services** (definition + counter, permanent). Warehouse holds only a volatile RAM cache (NOORDER). |
| Suspend loses cache? | **Yes (NOORDER).** No (ORDER). |
| Gap-free? | No. Gaps from lost cache batches (NOORDER) and rollbacks (both modes). |
| Unique? | Yes. Never repeats. |
| Independent of tables? | Yes — dropping a table does NOT drop its sequence. |
| Cross-table? | Yes — one sequence can serve multiple tables. |
| CURRVAL? | Not supported in Snowflake. |
| Negative increment? | Yes (`INCREMENT = -1`). |
| CYCLE/wrap-around? | No — errors on overflow. |
| Needs running warehouse? | Only to call NEXTVAL. Sequence exists without one. |

---
#### Reference: Sequences in Oracle (Gap Behavior)

Oracle sequences are **NOT gap-free** — same as Snowflake. Gaps occur for two reasons:

| Cause of Gap | Explanation |
|-------------|-------------|
| **Caching** | Oracle pre-allocates a batch of values in memory (default CACHE 20). If the instance shuts down or restarts, unused cached values are lost permanently. |
| **Rollbacks** | If a transaction calls `NEXTVAL` but then rolls back, the assigned number is consumed and never reused. |

**How Oracle caching works:**
```text
  CACHE 20 → Oracle pre-generates 20 values in memory

  Instance allocates: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

  Used so far: 1, 2, 3
  Instance crashes → values 4–20 are lost
  After restart → next value is 21 (gap of 4–20)
```

**Oracle vs Snowflake sequence comparison:**

| Behavior | Oracle | Snowflake |
|----------|--------|-----------|
| Gap-free? | No | No |
| Caching? | Yes (configurable: CACHE / NOCACHE) | Yes (NOORDER uses batches; ORDER does not) |
| Lost values on shutdown? | Yes (cached values lost) | Yes (NOORDER batches lost) |
| Rollback consumes value? | Yes | Yes |
| CURRVAL supported? | Yes | No |
| CYCLE (wrap-around)? | Yes | No |

**Key takeaway:** Neither Oracle nor Snowflake guarantees gap-free sequences. This is by design — enforcing gap-freedom would require serialization, destroying performance under concurrency.

---
## 4. Indexing in Snowflake

Snowflake **does NOT support traditional B-tree indexes** (no CREATE INDEX on regular tables). Instead, it uses a columnar micro-partition architecture with automatic optimizations that replace the need for indexes.

---

### 4.1 Why No Traditional Indexes?

Snowflake stores data in **micro-partitions** — immutable, compressed columnar files (50–500 MB each). Each micro-partition stores metadata about its contents (min value, max value, distinct count per column). When a query filters data, Snowflake reads this metadata and **skips entire partitions** whose min/max range doesn't match the filter. This is called **partition pruning** — it achieves the same goal as an index (skip irrelevant data) without maintaining a separate index structure.

---

### 4.2 Clustering Keys (Index Alternative #1)

**What it does:** Defines the physical sort order of data across micro-partitions. Rows with similar clustering key values are stored together, making min/max ranges narrow and non-overlapping — which maximizes partition pruning effectiveness.

**Why it exists:** Over time, as data is loaded in random order, micro-partition ranges overlap heavily. Clustering keys tell Snowflake to periodically re-organize (recluster) the data so that similar values end up in the same partitions.

```sql
-- Add clustering key to existing table
ALTER TABLE sales CLUSTER BY (sale_date, region);

-- Create table with clustering key
CREATE TABLE large_events (
    event_id VARCHAR,
    event_date DATE,
    category VARCHAR,
    payload VARIANT
) CLUSTER BY (event_date, category);

-- Check clustering efficiency
SELECT SYSTEM$CLUSTERING_INFORMATION('sales', '(sale_date, region)');

-- Drop clustering key
ALTER TABLE sales DROP CLUSTERING KEY;
```

**When to use:** Tables > 1 TB with queries that frequently filter on specific columns.

**Cost:** Automatic Clustering runs in the background using serverless compute credits.

---

### 4.3 Search Optimization Service (Index Alternative #2)

**What it does:** Creates internal **search access structures** (similar to inverted indexes) that accelerate highly selective queries — point lookups, substring searches, and semi-structured/geospatial queries.

**Why it exists:** Partition pruning works well for range queries (dates, ordered values) but poorly for **high-cardinality point lookups** (e.g., find one email out of 100M rows). Search Optimization builds dedicated lookup structures for these patterns.

```sql
-- Enable on entire table
ALTER TABLE customers ADD SEARCH OPTIMIZATION;

-- Enable on specific columns and query types
ALTER TABLE customers ADD SEARCH OPTIMIZATION
  ON EQUALITY(email),          -- WHERE email = 'x@y.com'
  ON SUBSTRING(name),          -- WHERE name LIKE '%smith%'
  ON GEO(location);            -- ST_DISTANCE, ST_WITHIN queries

-- Check status
DESCRIBE SEARCH OPTIMIZATION ON customers;

-- Remove
ALTER TABLE customers DROP SEARCH OPTIMIZATION;
```

**Accelerates these query patterns:**

| Query Type | Example |
|------------|---------|
| Equality (point lookup) | `WHERE email = 'user@company.com'` |
| LIKE / ILIKE (substring) | `WHERE name ILIKE '%smith%'` |
| Semi-structured (VARIANT) | `WHERE payload:status = 'active'` |
| Geospatial | `WHERE ST_DISTANCE(location, point) < 1000` |
| IN lists | `WHERE id IN (1, 2, 3, 4, 5)` |

**When to use:** Selective queries on high-cardinality columns where partition pruning alone is insufficient.

**Cost:** Additional storage for search structures + serverless maintenance compute.

---

### 4.4 Hybrid Table Indexes (True Indexes)

**What it does:** Provides actual **row-store B-tree indexes** (primary key, unique, and secondary) — the only true indexes in Snowflake.

**Why it exists:** Hybrid Tables serve OLTP workloads that require millisecond point lookups. A row-store index directly maps key values to row locations, enabling instant access without scanning any partitions.

```sql
CREATE HYBRID TABLE products (
    product_id INT PRIMARY KEY,           -- automatic primary index
    sku VARCHAR UNIQUE,                    -- automatic unique index
    name VARCHAR,
    category VARCHAR,
    price DECIMAL(10,2),
    INDEX idx_category (category),         -- secondary index
    INDEX idx_price_cat (category, price)  -- composite secondary index
);

-- Uses indexes automatically
SELECT * FROM products WHERE sku = 'SKU-12345';       -- unique index
SELECT * FROM products WHERE category = 'Electronics'; -- secondary index
```

**When to use:** OLTP workloads requiring millisecond single-row access.

**Cost:** Built into Hybrid Table — no extra billing, but Hybrid Tables have other trade-offs (no clustering keys, 1-day Time Travel limit).

---

### 4.5 Comparison

| Feature | What It Is | Table Type | Best For | Cost |
|---------|-----------|-----------|----------|------|
| Clustering Keys | Physical sort order across partitions | Regular tables | Range filters on large tables | Serverless reclustering |
| Search Optimization | Internal lookup structures | Regular tables | Point lookups, substring, geo | Storage + serverless maintenance |
| Hybrid Table Indexes | True B-tree row-store indexes | Hybrid tables only | OLTP millisecond access | Built into Hybrid Table |

---

### 4.6 Decision Guide

| Query Pattern | Use |
|---------------|-----|
| `WHERE date BETWEEN x AND y` (range scan on sorted data) | Clustering Key |
| `WHERE email = 'exact_value'` (point lookup) | Search Optimization |
| `WHERE name LIKE '%substring%'` | Search Optimization |
| `WHERE variant_col:key = 'value'` | Search Optimization |
| `WHERE pk = 'value'` with ms latency requirement | Hybrid Table Index |
| General large-table analytics with filters | Clustering Key |

---
## 5. Quick Reference

```text
┌──────────────────────────────────────────────────────────────────────┐
│                        TABLE TYPES                                   │
├───────────────┬────────────────┬────────────┬───────────────────────┤
│ Permanent     │ 90d TT + 7d FS │ Persistent │ Production            │
│ Transient     │ 1d TT, no FS   │ Persistent │ Staging/ETL           │
│ Temporary     │ 1d TT, no FS   │ Session    │ Scratch               │
│ External      │ None           │ Metadata   │ Data lake queries     │
│ Iceberg       │ Yes, no FS     │ Persistent │ Open-format interop   │
│ Event         │ Yes + FS       │ Persistent │ Logs/telemetry        │
│ Dynamic       │ Yes + FS       │ Persistent │ Auto pipelines        │
│ Hybrid        │ Yes + FS       │ Persistent │ OLTP + OLAP           │
│ Interactive   │ None           │ Session    │ App state             │
├───────────────┴────────────────┴────────────┴───────────────────────┤
│                        VIEW TYPES                                    │
├───────────────┬──────────────────────────────────────────────────────┤
│ Standard      │ Saved query, always fresh                            │
│ Materialized  │ Cached result, auto-maintained                       │
│ Secure        │ Hidden definition, safe for sharing                  │
│ Temporary     │ Session-scoped, auto-dropped                         │
│ Semantic      │ AI/Analyst metadata layer                            │
├───────────────┴──────────────────────────────────────────────────────┤
│                        INDEXING                                      │
├───────────────┬──────────────────────────────────────────────────────┤
│ Clustering Key│ Physical sort → partition pruning                    │
│ Search Opt.   │ Internal structures → point/geo lookups              │
│ Hybrid Index  │ True row-store indexes (PK + secondary)              │
└───────────────┴──────────────────────────────────────────────────────┘
```

---
*End of notes.*